In [16]:
from __future__ import annotations

import numpy
import random
import pandas
import time
import os
from deap import creator
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
from scipy.stats import pointbiserialr

from training_config import TrainingConfig
from multi_objective_training import MultiObjectiveTraining
from single_objective_training import SingleObjectiveTraining
from evaluation_utils import (
    get_continuous_columns, get_dummy_columns,
    build_model_package, evaluate_and_save,
    compute_stability_matrix, plot_stability_heatmap, plot_feature_frequency,
)

ImportError: cannot import name 'compute_stability_matrix' from 'evaluation_utils' (D:\RoutePatch\GitWorking\SiCo-MOGA\evaluation_utils.py)

In [ ]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    numpy.random.seed(seed)

In [ ]:
CSV_TRAIN_PATH: str = "hmeq/hmeq_preprocessed_train_data.csv"
CSV_TEST_PATH: str = "hmeq/hmeq_preprocessed_test_data.csv"
TARGET_COLUMN: str = "BAD"

RESULT_PATH: str = time.strftime("%Y-%m-%d_%H-%M-%S")
RESULT_PATH_MULTI: str = f"{RESULT_PATH}\\multi"
RESULT_PATH_SINGLE: str = f"{RESULT_PATH}\\single"
RESULT_PATH_EVAL: str = f"{RESULT_PATH}\\evaluation"

In [ ]:
# Load train set
df_train: pandas.DataFrame = pandas.read_csv(CSV_TRAIN_PATH)

# Split data into training and validation sets
X_search_pandas: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_search_pandas: pandas.Series = df_train[TARGET_COLUMN]

X_search: numpy.ndarray = numpy.ascontiguousarray(X_search_pandas.to_numpy(), dtype=numpy.float32)
y_search: numpy.ndarray = numpy.ascontiguousarray(y_search_pandas.to_numpy(), dtype=numpy.float32)

feature_names: list[str] = list(X_search_pandas.columns)

cv: StratifiedKFold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [ ]:
# Load test set
df_test: pandas.DataFrame = pandas.read_csv(CSV_TEST_PATH)

y_test: pandas.Series = df_test[TARGET_COLUMN]
X_test: pandas.DataFrame = df_test.drop(columns=[TARGET_COLUMN])

In [ ]:
# Pre-compute folds and scaling
fold_indices: list[tuple[numpy.ndarray, numpy.ndarray]] = list(cv.split(X_search, y_search))

X_train_scaled_folds: list[numpy.ndarray] = []
X_val_scaled_folds: list[numpy.ndarray] = []
y_train_folds: list[numpy.ndarray] = []
y_val_folds: list[numpy.ndarray] = []

n_splits: int = len(fold_indices)
n_features: int = len(feature_names)
corr_matrix: numpy.ndarray = numpy.zeros((n_splits, n_features), dtype=float)
for fold_idx, (train_idx, val_idx) in enumerate(fold_indices):
    X_fold_train: numpy.ndarray = X_search[train_idx]
    X_fold_val: numpy.ndarray = X_search[val_idx]
    y_fold_train: numpy.ndarray = y_search[train_idx]
    y_fold_val: numpy.ndarray = y_search[val_idx]

    # Pre-scale (all features at once)
    scaler: StandardScaler = StandardScaler()
    X_train_scaled_folds.append(scaler.fit_transform(X_fold_train))
    X_val_scaled_folds.append(scaler.transform(X_fold_val))
    y_train_folds.append(y_fold_train)
    y_val_folds.append(y_fold_val)

    # Calculate Point-biserial correlation coefficients (target value is binary and the input variables are continuous)
    # Calculate the average correlation for each feature across all fold
    for col_idx in range(X_fold_train.shape[1]):
        feature_col: numpy.ndarray = X_fold_train[:, col_idx]
        unique_vals: int = len(numpy.unique(feature_col))

        if unique_vals <= 1:
            corr: float = 0.0
        elif unique_vals == 2:
            corr: float = matthews_corrcoef(y_fold_train, feature_col)
        else:
            corr, _ = pointbiserialr(y_fold_train, feature_col)

        corr_matrix[fold_idx, col_idx] = corr

In [ ]:
training_results_multi: dict[int, list[list[creator.Individual]]] = {}
training_results_single: dict[int, list[creator.IndividualSingle]] = {}

seeds: list[int] = [42, 123, 5678]
for s in seeds:
    set_seed(s)
    multi_objective_training: MultiObjectiveTraining = MultiObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_MULTI),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds,
        corr_matrix=corr_matrix,
    )
    pareto_front: list[creator.Individual] = multi_objective_training.run()
    training_results_multi[s] = pareto_front
    multi_objective_training.clear_cache()

    set_seed(s)
    single_objective_training: SingleObjectiveTraining = SingleObjectiveTraining(
        config=TrainingConfig(seed=s, result_directory=RESULT_PATH_SINGLE),
        feature_names=feature_names,
        fold_indices=fold_indices,
        X_train_scaled_folds=X_train_scaled_folds,
        X_val_scaled_folds=X_val_scaled_folds,
        y_train_folds=y_train_folds,
        y_val_folds=y_val_folds)
    best_indi: creator.IndividualSingle = single_objective_training.run()
    training_results_single[s] = best_indi
    single_objective_training.clear_cache()

## Evaluation: Single vs Multi-Objective Robustness Comparison

For each seed we:
1. Retrain a final LogisticRegression on the **full training set** using features selected by the GA.
   - Multi-objective: the Pareto individual with the **highest sign-consistency** score.
   - Single-objective: the best individual from the Hall of Fame.
2. Evaluate on clean test data and under increasing noise levels:
   - **Gaussian noise + mean shift** on continuous features
   - **Bit-flip noise** on dummy (binary) features
3. Save CSV results and comparison plots to the result directory.

In [ ]:
# Detect column types automatically from the training DataFrame
X_train_df: pandas.DataFrame = df_train.drop(columns=[TARGET_COLUMN])
y_train_series: pandas.Series = df_train[TARGET_COLUMN]

continuous_cols: list[str] = get_continuous_columns(X_train_df)
dummy_cols: list[str] = get_dummy_columns(X_train_df)

# Training-set std for proportional noise scaling
train_std: pandas.Series = X_train_df.std()

print(f"Continuous features: {len(continuous_cols)}")
print(f"Dummy features:     {len(dummy_cols)}")

In [ ]:
# Multi-objective: pick the Pareto individual with the highest sign-consistency
evaluation_results: dict[int, pandas.DataFrame] = {}

for s in seeds:
    set_seed(s)

    # --- Multi-objective: select max sign-consistency individual from Pareto front ---
    pareto_front = training_results_multi[s]
    best_multi_ind = max(pareto_front, key=lambda ind: ind.fitness.values[1])
    print(f"Seed {s} | Selected multi-objective individual: "
          f"AUC={best_multi_ind.fitness.values[0]:.4f}, "
          f"SignConsistency={best_multi_ind.fitness.values[1]:.4f}, "
          f"Features={sum(best_multi_ind)}")

    # --- Build final model packages on full training set ---
    model_pkg_multi = build_model_package(
        best_multi_ind, feature_names, X_train_df, y_train_series, seed=s)
    model_pkg_single = build_model_package(
        training_results_single[s], feature_names, X_train_df, y_train_series, seed=s)

    # --- Run noise evaluation and save plots/CSV ---
    eval_df = evaluate_and_save(
        seed=s,
        model_pkg_multi=model_pkg_multi,
        model_pkg_single=model_pkg_single,
        X_test=X_test,
        y_test=y_test,
        train_std=train_std,
        continuous_cols=continuous_cols,
        dummy_cols=dummy_cols,
        result_directory=RESULT_PATH_EVAL,
        use_roc_auc=True,
    )
    evaluation_results[s] = eval_df

print("\nEvaluation complete. Results saved to:", RESULT_PATH_EVAL)

## Feature Selection Stability (Jaccard Similarity)

Compare how consistently each method selects the same features across different random seeds.
- **Jaccard similarity** = |intersection| / |union| between two feature sets (1.0 = identical, 0.0 = no overlap).
- A **pairwise heatmap** shows seed-to-seed stability for multi-objective vs single-objective.
- A **feature frequency bar chart** shows which features are consistently selected across seeds.

In [ ]:
# Collect the best individual per seed for both methods
best_multi_by_seed: dict[int, list[int]] = {}
best_single_by_seed: dict[int, list[int]] = {}

for s in seeds:
    # Multi-objective: max sign-consistency (same selection as evaluation)
    pareto_front: list[creator.Individual] = training_results_multi[s]
    best_multi_ind: creator.Individual = max(pareto_front, key=lambda ind: ind.fitness.values[1])
    best_multi_by_seed[s]: list[int] = list(best_multi_ind)

    # Single-objective: best individual from HoF
    best_single_by_seed[s]: list[int] = list(training_results_single[s])

# Compute pairwise Jaccard stability matrices
seeds_multi, matrix_multi = compute_stability_matrix(best_multi_by_seed, feature_names)
seeds_single, matrix_single = compute_stability_matrix(best_single_by_seed, feature_names)

# Print summary
print("=== Feature Selection Stability (Jaccard Similarity) ===\n")
print("Multi-Objective (SiCo-MOGA) pairwise Jaccard:")
for i, si in enumerate(seeds_multi):
    for j, sj in enumerate(seeds_multi):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_multi[i, j]:.3f}")

print("\nSingle-Objective (AUC-only GA) pairwise Jaccard:")
for i, si in enumerate(seeds_single):
    for j, sj in enumerate(seeds_single):
        if j > i:
            print(f"  Seed {si} vs Seed {sj}: {matrix_single[i, j]:.3f}")

# Save plots
stability_dir: str = os.path.join(RESULT_PATH_EVAL, "stability")

plot_stability_heatmap(
    seeds_multi, matrix_multi,
    seeds_single, matrix_single,
    os.path.join(stability_dir, "jaccard_heatmap.png"))

plot_feature_frequency(
    best_multi_by_seed, feature_names,
    "Multi-Objective (SiCo-MOGA)",
    os.path.join(stability_dir, "feature_frequency_multi.png"))

plot_feature_frequency(
    best_single_by_seed, feature_names,
    "Single-Objective (AUC-only GA)",
    os.path.join(stability_dir, "feature_frequency_single.png"))

print(f"\nStability plots saved to: {stability_dir}")